#Import Information about Project

##Column Name Definitions
* Report Number: The specific ID of the accident
* Event Date: The date of the accident
* City: Where the accident occured (city)
* State: Where the accident occured (state)
* Airport: If the accident occured during take off or landing, this identifies the airport in which it occured
* Event Type:
* Aircraft Damage: The amount of damage that the plane took during the accident
* Flight Phase: The phase of flight in which the accident occured, e.g. take off, landing, manuver, cruise, etc.
* Aircraft Make: The manufacturer, brand, or company that made the aircraft in the accident
* Aircraft Model: Specific product line, design, or series created by a manufacturer
* Operator Code: The company or person operating the aircraft
* Flight Conduct Code:
* Aircraft Registration Number: Number that identifies a specific airplane
* Total Fatalities: The amount of people killed in the accident
* Total Injuries: The amount of people injured in the accident
* Engine Make: The brand, company, or manufacturer of the engine
* Engine Model: The specific design or series of engines created by the manufacturer
* Engine Group Code:
* PIC Certification Type:

##Accident vs. Incident Definition

In [ ]:
import pandas as pd

df = pd.read_csv("faa_aids_all.csv", low_memory=False)

print("Original Dataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nDataset Information:")
df.info()

exact_duplicates = df.duplicated().sum()

print("\nExact Duplicate Rows:", exact_duplicates)

report_counts = df['rprt_nbr'].value_counts()

print("\nTotal Rows:", len(df))
print("Unique Reports:", df['rprt_nbr'].nunique())

print(
    "Reports Appearing Multiple Times:",
    (report_counts > 1).sum()
)

df_clean = df.drop_duplicates().copy()

print("\nOriginal Rows:", len(df))
print("Rows After Removing Duplicates:", len(df_clean))
print("Rows Removed:", len(df) - len(df_clean))
print("Remaining Exact Duplicates:", df_clean.duplicated().sum())

df_clean['event_lcl_date'] = pd.to_datetime(
    df_clean['event_lcl_date'],
    format='%Y %b %d',
    errors='coerce'
)

print("\nDate Data Type:", df_clean['event_lcl_date'].dtype)
print("Missing Dates:", df_clean['event_lcl_date'].isna().sum())
print("Earliest Event:", df_clean['event_lcl_date'].min())
print("Latest Event:", df_clean['event_lcl_date'].max())

missing_summary = pd.DataFrame({
    'Missing_Count': df_clean.isnull().sum(),
    'Missing_Percentage': (
        df_clean.isnull().mean() * 100
    ).round(2)
})

missing_summary = missing_summary[
    missing_summary['Missing_Count'] > 0
].sort_values(
    'Missing_Percentage',
    ascending=False
)

print("\nMissing Value Summary:")
display(missing_summary)

damage_counts = df_clean['acft_dmg_code'].value_counts(
    dropna=False
)

damage_summary = pd.DataFrame({
    'Count': damage_counts,
    'Percentage': (
        damage_counts / len(df_clean) * 100
    ).round(2)
})

print("\nAircraft Damage Distribution:")
display(damage_summary)

#selecting the columns directly related to the research question
project_columns = [
    'rprt_nbr',
    'event_type_code',
    'flt_phase_code',
    'pic_cert_type_code',
    'acft_make',
    'acft_model',
    'acft_dmg_code',
    'inj_total_invlvd_fatal_qty',
    'inj_total_invlvd_non_fatal_qty'
]

project_df = df_clean[project_columns].copy()

print("\nProject Dataset Shape:", project_df.shape)

print("\nProject Dataset Preview:")
display(project_df.head())


#create classification labels for injury and fatality outcomes

#fatalities: 0 = None, 1 = One or more
project_df['fatality_class'] = (
    project_df['inj_total_invlvd_fatal_qty']
    .apply(lambda x: 1 if x > 0 else 0 if pd.notna(x) else pd.NA)
    .astype('Int64')
)

#nonfatal injuries: 0 = None, 1 = One or more
project_df['nonfatal_injury_class'] = (
    project_df['inj_total_invlvd_non_fatal_qty']
    .apply(lambda x: 1 if x > 0 else 0 if pd.notna(x) else pd.NA)
    .astype('Int64')
)

print("Fatality Classification:")
display(project_df['fatality_class'].value_counts(dropna=False))

print("\nNonfatal Injury Classification:")
display(project_df['nonfatal_injury_class'].value_counts(dropna=False))

#prepare the three classification targets

#remove rows with missing or unknown aircraft damage
damage_df = project_df[
    project_df['acft_dmg_code'].notna() &
    (project_df['acft_dmg_code'] != 'Unknown')
].copy()

#keep only the three usable damage categories
damage_df = damage_df[
    damage_df['acft_dmg_code'].isin(['Minor', 'Substantial', 'Destroyed'])
]

print("Aircraft Damage Target:")
display(damage_df['acft_dmg_code'].value_counts())

print("\nFatality Target:")
display(project_df['fatality_class'].value_counts(dropna=False))

print("\nNonfatal Injury Target:")
display(project_df['nonfatal_injury_class'].value_counts(dropna=False))

print("\nRows available for damage classification:", len(damage_df))

# prepare the predictor columns for machine learning
feature_columns = [
    'event_type_code',
    'flt_phase_code',
    'pic_cert_type_code',
    'acft_make',
    'acft_model'
]

#prepare the damage classification dataset
damage_model_df = damage_df[
    feature_columns + ['acft_dmg_code', 'rprt_nbr']
].copy()

#prepare the fatality classification dataset
fatality_model_df = project_df[
    feature_columns + ['fatality_class', 'rprt_nbr']
].dropna(subset=['fatality_class']).copy()

#prepare the nonfatal injury classification dataset
nonfatal_model_df = project_df[
    feature_columns + ['nonfatal_injury_class', 'rprt_nbr']
].dropna(subset=['nonfatal_injury_class']).copy()

#replace missing predictor values with "Unknown"
for dataset in [damage_model_df, fatality_model_df, nonfatal_model_df]:
    dataset[feature_columns] = dataset[feature_columns].fillna('Unknown')

print("Damage Classification Rows:", len(damage_model_df))
print("Fatality Classification Rows:", len(fatality_model_df))
print("Nonfatal Injury Classification Rows:", len(nonfatal_model_df))

print("\nPredictor Columns:")
print(feature_columns)

print("\nPrepared Damage Dataset:")
display(damage_model_df.head())

Original Dataset Shape: (251290, 19)

Column Names:
['rprt_nbr', 'event_lcl_date', 'loc_city_name', 'loc_state_code', 'loc_arpt_name', 'event_type_code', 'acft_dmg_code', 'flt_phase_code', 'acft_make', 'acft_model', 'oprtr_code', 'fltcndct_code', 'regist_nbr', 'inj_total_invlvd_fatal_qty', 'inj_total_invlvd_non_fatal_qty', 'eng_make_name', 'eng_model_name', 'eng_grp_mkmd_code', 'pic_cert_type_code']

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 251290 entries, 0 to 251289
Data columns (total 19 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   rprt_nbr                        251290 non-null  object 
 1   event_lcl_date                  251290 non-null  object 
 2   loc_city_name                   248945 non-null  object 
 3   loc_state_code                  250370 non-null  object 
 4   loc_arpt_name                   176291 non-null  object 
 5   event_type_code          

,Missing_Count,Missing_Percentage
oprtr_code,201725,81.63
pic_cert_type_code,194200,78.59
eng_grp_mkmd_code,162656,65.82
eng_model_name,121281,49.08
eng_make_name,120230,48.65
loc_arpt_name,73795,29.86
acft_dmg_code,40697,16.47
fltcndct_code,28051,11.35
flt_phase_code,6916,2.80
inj_total_invlvd_fatal_qty,5435,2.20



Aircraft Damage Distribution:


,Count,Percentage
acft_dmg_code,,
Substantial,92720,37.52
Minor,81099,32.82
NaN,40697,16.47
Destroyed,32286,13.07
Unknown,314,0.13



Project Dataset Shape: (247116, 9)

Project Dataset Preview:


,rprt_nbr,event_type_code,flt_phase_code,pic_cert_type_code,acft_make,acft_model,acft_dmg_code,inj_total_invlvd_fatal_qty,inj_total_invlvd_non_fatal_qty
0,20260930012829A,Accident,Landing: Touchdown,AIRLINE TRANSPORT,CESSNA,CE-525,Destroyed,0.0,0.0
1,20260930013249I,Incident,Ground/Ramp/Ma,FLIGHT INSTRUCTOR,ENSTROM HELICOPTER CORP,480,Substantial,0.0,0.0
2,20260930013249I,Incident,Ground/Ramp/Ma,COMMERCIAL PILOT,ENSTROM HELICOPTER CORP,480,Substantial,0.0,0.0
3,20260929013129I,Incident,Takeoff: Climb Out,COMMERCIAL PILOT,BELLANCA,7ACA,Minor,0.0,0.0
4,20260929013129I,Incident,Takeoff: Climb Out,FLIGHT INSTRUCTOR,BELLANCA,7ACA,Minor,0.0,0.0


... (8 outputs left out)


Nonfatal Injury Target:


,count
nonfatal_injury_class,
0,231942
1,9739
<NA>,5435



Rows available for damage classification: 206105
Damage Classification Rows: 206105
Fatality Classification Rows: 241681
Nonfatal Injury Classification Rows: 241681

Predictor Columns:
['event_type_code', 'flt_phase_code', 'pic_cert_type_code', 'acft_make', 'acft_model']

Prepared Damage Dataset:


,event_type_code,flt_phase_code,pic_cert_type_code,acft_make,acft_model,acft_dmg_code,rprt_nbr
0,Accident,Landing: Touchdown,AIRLINE TRANSPORT,CESSNA,CE-525,Destroyed,20260930012829A
1,Incident,Ground/Ramp/Ma,FLIGHT INSTRUCTOR,ENSTROM HELICOPTER CORP,480,Substantial,20260930013249I
2,Incident,Ground/Ramp/Ma,COMMERCIAL PILOT,ENSTROM HELICOPTER CORP,480,Substantial,20260930013249I
3,Incident,Takeoff: Climb Out,COMMERCIAL PILOT,BELLANCA,7ACA,Minor,20260929013129I
4,Incident,Takeoff: Climb Out,FLIGHT INSTRUCTOR,BELLANCA,7ACA,Minor,20260929013129I
